# Entrenamiento de sistemas de IA mediante Aprendizaje por Refuerzo para tareas de razonamiento  


## Modelo parametrizado

Un modelo es una función

$$
f_\theta: X \to Y
$$

que recibe una entrada $x \in X$ y produce una salida $f_\theta(x) \in Y$.  
El subindice $\theta$ indica que el comportamiento de la función depende de un conjunto de parametros, es decir, depende de $\theta$, y por lo tanto si $\theta$ cambia entonces el modelo cambia, lo cual es basicamente lo que se ajusta.

- $X$: espacio de entradas (por ejemplo, vectores numericos, imagenes, o secuencias de tokens).
- $Y$: espacio de salidas (por ejemplo, una clase, un numero, una secuencia de tokens o una accion).
- $\theta$: parametros ajustables (pesos y sesgos), que son los que se van moviendo.

Idea clave: “Aprender” significa escoger $\theta$ para que $f_\theta$ haga lo que queremos, segun un criterio medible, osea algo que si podamos cuantificar y no solo decir “se ve bien”.


## Datos y objetivo de entrenamiento

En muchos casos se dispone de ejemplos:

$$
D=\{(x_i,y_i)\}_{i=1}^n
$$

donde $x_i$ es la entrada y $y_i$ es la salida deseada (la “respuesta correcta” o “etiqueta”). Este es el escenario tipico del aprendizaje supervisado, por que ya se conoce la salida correcta antes de entrenar, y eso guia el ajuste.

Para medir qué tan bien se comporta el modelo se define una perdida (o error), por ejemplo $\ell(\hat{y}, y)$, donde $\hat{y} = f_{\theta}(x)$ es la predicción del modelo. La perdida total suele ser un promedio, aunque tambien se puede pensar como suma normalizada:

$$
L(\theta) = \frac{1}{n}\sum_{i=1}^{n} \ell\big(f_{\theta}(x_i), y_i\big).
$$

El entrenamiento busca los parametros que minimizen esa perdida, o dicho de otra forma, se quiere que el error sea lo mas chico posible, entonces se define:

$$
\theta^{*} = \arg\min_{\theta} L(\theta).
$$


## Optimización: descenso por gradiente

Queremos minimizar una función de perdida $L(\theta)$ (donde $\theta$ son los parametros).  
La idea del descenso por gradiente sale de una aproximación de primer orden (Taylor) alrededor de $\theta$, osea una aproximación local que nos dice como cambia $L$ si movemos un poco $\theta$.

### Aproximación local (Taylor de primer orden)

Para un cambio pequeño $\Delta\theta$, se tiene:

$$
L(\theta + \Delta\theta)
\approx
L(\theta) + \nabla_{\theta}L(\theta)^\top \Delta\theta.
$$

Esto dice que, cerca de $\theta$, la perdida cambia casi linealmente, y que la dirección que más aumenta $L$ es el gradiente $\nabla_\theta L(\theta)$ (por eso se le toma como referencia, aun que no es magia, es por Taylor).

### Elegir una dirección que disminuya $L$

Si elegimos $\Delta\theta$ en la dirección opuesta al gradiente:

$$
\Delta\theta = -\eta \,\nabla_{\theta}L(\theta), \qquad \eta>0,
$$

entonces al sustituir en la aproximación:

$$
L(\theta + \Delta\theta)
\approx
L(\theta) + \nabla_{\theta}L(\theta)^\top\big(-\eta \nabla_{\theta}L(\theta)\big)
=
L(\theta) - \eta \,\|\nabla_{\theta}L(\theta)\|^2.
$$

Como $\|\nabla_{\theta}L(\theta)\|^2 \ge 0$ y $\eta>0$, esto sugiere que (si $\eta$ es suficentemente pequeña) la perdida disminuye, en promedio y localmente, es decir, en esa vecindad.

De aqui sale la actualización estándar. Para evitar ambiguedad, se escribe por iteraciones:

$$
\theta^{(k+1)} = \theta^{(k)} - \eta\,\nabla_\theta L\big(\theta^{(k)}\big).
$$

donde:

- $\nabla_\theta L(\theta)$ es el vector de derivadas parciales de $L$ respecto a cada parametro.
- $\eta>0$ es la tasa de aprendizaje (tamaño del paso), y si se elige mal puede hacer que no baje o que oscile.


## Regla de la cadena y retropropagación (idea matematica)

En redes profundas, $L(\theta)$ es una composición de muchas funciones, por eso el gradiente no se calcula “a mano” termino por termino, si no aplicando la regla de la cadena de forma sistematica.

Para ver el mecanismo, consideremos una composición simple:

$$
f_{\theta}(x) = g_{\theta_2}\big(h_{\theta_1}(x)\big),
\qquad
\hat{y}=f_{\theta}(x),
\qquad
L(\theta)=\ell(\hat{y},y).
$$

Definimos la variable intermedia:

$$
z = h_{\theta_1}(x),
\qquad
\hat{y} = g_{\theta_2}(z).
$$

### Gradiente respecto a $\theta_2$

Por regla de la cadena:

$$
\nabla_{\theta_2} L
=
\frac{\partial L}{\partial \hat{y}}
\;\frac{\partial \hat{y}}{\partial \theta_2}
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{\theta_2} g_{\theta_2}(z)\Big).
$$

La idea es: primero ves como cambia la perdida con la salida $\hat{y}$, y luego como cambia $\hat{y}$ con $\theta_2$, y se combinan.

### Gradiente respecto a $\theta_1$

Aqui $L$ depende de $\theta_1$ a través de $z=h_{\theta_1}(x)$, entonces:

$$
\nabla_{\theta_1} L
=
\frac{\partial L}{\partial z}\;\frac{\partial z}{\partial \theta_1}.
$$

Pero $\frac{\partial L}{\partial z}$ no es directo, por que $L$ pasa por $\hat{y}$:

$$
\frac{\partial L}{\partial z}
=
\frac{\partial L}{\partial \hat{y}}\;\frac{\partial \hat{y}}{\partial z}
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{z} g_{\theta_2}(z)\Big).
$$

Por lo tanto:

$$
\nabla_{\theta_1} L
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{z} g_{\theta_2}(z)\Big)\Big(\nabla_{\theta_1} h_{\theta_1}(x)\Big).
$$

En palabras: cada bloque aporta su derivada y se van multiplicando “hacia atras” (de la salida hacia la entrada).  
Eso es basicamente backpropagation, osea la forma eficiente de calcular todos los gradientes sin recalcular todo desde cero.


## Neurona (unidad básica) y por qué necesitamos no linealidad

Una **neurona** (o unidad) recibe una entrada $x \in \mathbb{R}^d$ y calcula primero una combinación afín (lineal + sesgo):

$$
z = w^\top x + b,
$$

y luego aplica una función de activación (típicamente no lineal) para producir la salida:

$$
y = \sigma(z),
$$

donde:

- $w \in \mathbb{R}^d$ son los **pesos**,
- $b \in \mathbb{R}$ es el **sesgo**,
- $\sigma:\mathbb{R}\to\mathbb{R}$ es la **activación**.

Un ejemplo muy usado es ReLU:

$$
\sigma(z)=\max\{0,z\}.
$$

**Por qué es necesaria la no linealidad:**  
Si $\sigma$ fuera la identidad (es decir, $\sigma(z)=z$), entonces cada capa sería solo una transformación afín. Al componer varias capas afines, el resultado sigue siendo una transformación afín; por ejemplo, sin sesgos para simplificar,

$$
x \mapsto W_2(W_1x) = (W_2W_1)x.
$$

Esto significa que, aunque apiles muchas capas, el modelo no gana “potencia” expresiva: seguiría comportándose como un modelo lineal. En cambio, al introducir una activación no lineal $\sigma$, la composición deja de ser lineal y la red puede aproximar relaciones mucho más complejas.


## Red neuronal multicapa (MLP) como composición de capas

Una red *feed-forward* (MLP) con $L$ capas puede escribirse como una composición iterada. Definimos la entrada como

$$
h_0 = x,
$$

y para cada capa oculta $\ell = 1, \dots, L-1$:

$$
h_\ell = \sigma\big(W_\ell h_{\ell-1} + b_\ell\big).
$$

Finalmente, una forma común de la **capa de salida** (por ejemplo, en regresión) es:

$$
f_\theta(x)= W_L h_{L-1} + b_L.
$$

En conjunto, los parámetros del modelo son:

$$
\theta = \{(W_\ell, b_\ell)\}_{\ell=1}^{L}.
$$



## Representación de texto: tokens y embeddings

Un modelo de lenguaje no “entiende” letras directamente. Para poder trabajar con el texto, primero lo convierte en números. El proceso típico tiene tres pasos: **tokenización**, **vocabulario** y **embeddings**.

**Tokenización (pasar texto a tokens)**

Dado un texto, se divide en piezas llamadas **tokens**. En la práctica, los tokens suelen ser **sub-palabras** (no necesariamente palabras completas).  
Después, a cada token se le asigna un **índice entero**.

Si el texto queda representado por la secuencia de índices

$$
(t_1,t_2,\dots,t_n),
$$

entonces $t_i$ es el índice del token en la posición $i$.

**Vocabulario**

El vocabulario es el conjunto de todos los tokens que el modelo puede reconocer. Lo denotamos por $V$, y su tamaño es $|V|$.  
Cada token del vocabulario tiene un índice en $\{1,2,\dots,|V|\}$ (a veces se usa $\{0,1,\dots,|V|-1\}$, depende de la implementación).

**Embeddings**

Un índice por sí solo no contiene “información” útil para el modelo. Por eso, cada token se representa mediante un vector en $\mathbb{R}^d$.  
Para esto se aprende una matriz de embeddings

$$
E \in \mathbb{R}^{|V|\times d}.
$$

La fila $t$ de esta matriz es el embedding del token con índice $t$. Es decir:

$$
e_t = E[t] \in \mathbb{R}^d.
$$

Entonces, para la secuencia $(t_1,\dots,t_n)$, se obtienen los vectores

$$
e_{t_1}, e_{t_2}, \dots, e_{t_n}.
$$

Si se apilan como filas, se forma la matriz de entrada

$$
X =
\begin{bmatrix}
e_{t_1}\\
e_{t_2}\\
\vdots\\
e_{t_n}
\end{bmatrix}
\in \mathbb{R}^{n\times d}.
$$

Aquí $n$ es el número de tokens del texto y $d$ es la dimensión de cada embedding.

## La necesidad de “contexto” en secuencias

En lenguaje, el significado de una palabra depende del contexto. Por ejemplo, “banco” puede significar institución financiera o asiento. Un modelo necesita producir representaciones donde cada token incorpore información de otros tokens relevantes. Esto se logra con **atención**.


## Transformer: arquitectura basada en auto-atención

Un **Transformer** procesa una secuencia representada por

$$
X \in \mathbb{R}^{n\times d},
$$

donde $n$ es el número de tokens y $d$ es la dimensión del embedding.  
El Transformer está formado por bloques que se repiten, y la operación principal dentro de cada bloque es la **auto-atención** (*self-attention*).


**Construcción de $Q$, $K$ y $V$**

A partir de $X$ se construyen tres matrices mediante proyecciones lineales:

$$
Q = XW_Q,\qquad K = XW_K,\qquad V = XW_V,
$$

con

- $W_Q, W_K \in \mathbb{R}^{d\times d_k}$,
- $W_V \in \mathbb{R}^{d\times d_v}$,

por lo tanto

$$
Q, K \in \mathbb{R}^{n\times d_k},\qquad V \in \mathbb{R}^{n\times d_v}.
$$

**Intuición**
- $Q$ (*queries*) describe lo que cada token “pregunta” o “busca”.
- $K$ (*keys*) describe con qué “etiqueta” se presenta cada token.
- $V$ (*values*) es la información que se va a combinar para producir la salida.


**Puntajes de atención y pesos**

Primero se calcula una matriz de puntajes comparando cada query con cada key:

$$
S = \frac{QK^\top}{\sqrt{d_k}} \in \mathbb{R}^{n\times n}.
$$

El factor $\sqrt{d_k}$ se usa para evitar que los puntajes crezcan demasiado cuando $d_k$ es grande, lo cual ayuda a que la función *softmax* no quede saturada.

Después, se transforman los puntajes en **pesos de atención** aplicando *softmax* por filas:

$$
A = \mathrm{softmax}(S),
$$

es decir, para cada $i$:

$$
A_{ij}=\frac{e^{S_{ij}}}{\sum_{m=1}^{n} e^{S_{im}}}.
$$

Cada fila de $A$ suma 1, y $A_{ij}$ indica cuánto peso le da la posición $i$ a la posición $j$.



**Mezcla de valores (salida de atención)**

La salida de auto-atención se obtiene mezclando los valores con esos pesos:

$$
\mathrm{Attn}(X) = AV.
$$

**Interpretación:** para cada posición $i$, el vector de salida es una combinación de los vectores $V_1,\dots,V_n$.  
Los coeficientes de esa combinación están en la fila $i$ de $A$, es decir, en $A_{i1},\dots,A_{in}$.

## Multi-head attention (varias atenciones en paralelo)

En lugar de calcular una sola auto-atención, un Transformer usa **$H$ cabezas** (*heads*) en paralelo. La idea es que cada cabeza aprende una forma distinta de comparar tokens.

Para cada cabeza $h \in \{1,\dots,H\}$ se usan matrices de proyección diferentes:

$$
Q^{(h)} = XW_Q^{(h)},\qquad
K^{(h)} = XW_K^{(h)},\qquad
V^{(h)} = XW_V^{(h)}.
$$

Luego, la salida de la cabeza $h$ se define como:

$$
\mathrm{head}^{(h)} \;=\;
\mathrm{softmax}\!\left(\frac{Q^{(h)}(K^{(h)})^\top}{\sqrt{d_k}}\right)V^{(h)}.
$$

Finalmente, se concatenan las salidas de todas las cabezas y se aplica una proyección lineal:

$$
\mathrm{MHA}(X)
=
\mathrm{Concat}\!\big(\mathrm{head}^{(1)},\dots,\mathrm{head}^{(H)}\big)\,W_O.
$$

**Por qué usar varias cabezas:**  
Cada cabeza puede enfocarse en un tipo de relación diferente entre tokens (por ejemplo, dependencias locales, referencias a tokens lejanos, patrones sintácticos o relaciones más lógicas). Al combinar varias, el modelo obtiene una representación más rica.



**Positional encoding (cómo se representa el orden)**

La auto-atención, por sí sola, no tiene forma de saber qué token está antes o después: si se permuta el orden de los tokens, el mecanismo de atención no “se da cuenta” automáticamente.  
Por eso se agrega información de posición.

Una forma estándar es sumar a cada embedding un vector de posición $p_i \in \mathbb{R}^d$:

$$
\widetilde{X}_i = X_i + p_i,\qquad i=1,\dots,n.
$$

En forma matricial, si $P \in \mathbb{R}^{n\times d}$ apila los vectores de posición como filas, entonces:

$$
\widetilde{X} = X + P.
$$

Los vectores $p_i$ pueden ser **sinusoidales** (fijos) o **aprendidos** (parámetros entrenables). Lo importante es que el modelo reciba suficiente información para distinguir el orden de la secuencia.

## Bloque Transformer (estructura típica)

Un bloque Transformer estándar combina dos subcapas:

1) **Multi-head attention (MHA)**  
2) **MLP** (una red feed-forward aplicada token por token)

En la práctica se usan **normalización** y **conexiones residuales**. Una forma común (llamada *pre-norm*) se escribe así.

Primero, la salida de la subcapa de atención:

$$
H = X + \mathrm{MHA}\big(\mathrm{LN}(X)\big),
$$

después, la salida de la subcapa MLP:

$$
Y = H + \mathrm{MLP}\big(\mathrm{LN}(H)\big).
$$

Aquí:

- $\mathrm{LN}$ es **LayerNorm** (normalización por características).
- Las conexiones residuales (los términos $X + \cdot$ y $H + \cdot$) ayudan a estabilizar el entrenamiento en redes profundas, porque permiten que la información y los gradientes fluyan mejor a través de muchas capas.
- La MLP se aplica de forma **independiente en cada posición** (misma transformación para todos los tokens, pero sin mezclar posiciones).

## Modelos de lenguaje autoregresivos (qué optimizan)

Un modelo de lenguaje **autoregresivo** aprende a predecir el siguiente token usando todos los tokens anteriores como contexto.

Si el texto (ya tokenizado) es

$$
(t_1, t_2, \dots, t_n),
$$

el modelo define, para cada posición $i$, una distribución de probabilidad del token $t_i$ condicionada a los tokens previos:

$$
p_\theta\!\left(t_i \mid t_{<i}\right),
\qquad
\text{donde } t_{<i} = (t_1,\dots,t_{i-1}).
$$

### Función objetivo (máxima verosimilitud)

Durante el entrenamiento se busca que el modelo asigne alta probabilidad al token correcto en cada paso. Esto se hace maximizando la probabilidad conjunta de la secuencia:

$$
p_\theta(t_1,\dots,t_n)
=
\prod_{i=1}^{n} p_\theta\!\left(t_i \mid t_{<i}\right).
$$

Maximizar esta cantidad es equivalente a minimizar la **pérdida de entropía cruzada** (negative log-likelihood):

$$
L(\theta) = -\sum_{i=1}^{n} \log p_\theta\!\left(t_i \mid t_{<i}\right).
$$

En palabras: en cada posición $i$ el modelo “propone” probabilidades para el siguiente token, y se penaliza cuando el token verdadero tiene probabilidad baja.

Este objetivo hace que el modelo aprenda regularidades del lenguaje y produzca texto coherente.  
Pero, estrictamente, lo que optimiza sigue siendo **predecir tokens**. En tareas de planeación o razonamiento secuencial, a veces no basta con predecir: lo que interesa es **tomar decisiones** para alcanzar una meta (y ahí entra el aprendizaje por refuerzo).

## De predecir texto a elegir acciones

Hay tareas donde el sistema debe producir una **secuencia de decisiones**, y la calidad de esa secuencia se evalúa hasta el final (o con señales muy parciales durante el proceso). Por ejemplo:

- resolver un acertijo en varios pasos,
- planear una ruta,
- construir un procedimiento que satisfaga restricciones,
- generar una cadena de razonamiento que lleve a una respuesta correcta.

En estos casos, la retroalimentación natural no es “cuál era el siguiente token”, sino una **señal numérica de desempeño**: éxito o fracaso, distancia a una meta, penalización por violar reglas, costo por tiempo, etc.

Por eso conviene cambiar el punto de vista: en lugar de pensar en “predicción del siguiente token”, se piensa en un **agente** que elige acciones paso a paso dentro de un **entorno**, y recibe recompensas.  


## Aprendizaje por refuerzo: interacción agente–entorno

En aprendizaje por refuerzo (Reinforcement Learning, RL) no se parte de una lista fija de respuestas correctas $(x_i,y_i)$. En vez de eso, se considera un sistema que interactua con un entorno, y a partir de esa interacción se obtiene informacion para aprender.

La idea es describir un ciclo que sucede en pasos de tiempo $t=0,1,2,\dots$:

1. El agente observa un estado $S_t$.
2. El agente elije una acción $A_t$.
3. El entorno responde con:
   - un nuevo estado $S_{t+1}$,
   - y una recompensa $R_{t+1}$ (un numero real).

Este ciclo se repite hasta que termina el episodio (por ejemplo, por que se llegó a la meta o se acabaron los pasos disponibles).

## Formulación como Proceso de Decisión de Markov (MDP)

Un problema de RL se modela como:

$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma),
$$

donde:

- $\mathcal{S}$ es el conjunto de estados posibles.
- $\mathcal{A}$ es el conjunto de acciones posibles.
- $P(s' \mid s,a)$ es la probabilidad de transición: estando en $s$ y tomando la acción $a$, indica la probabilidad de pasar a $s'$.
- $R(s,a,s')$ es la recompensa asociada a la transición $s \to s'$ al ejecutar $a$.
- $\gamma \in [0,1)$ es el factor de descuento, que controla cuánto pesan las recompensas futuras frente a las inmediatas.

## Política (la “regla” que sigue el agente)

Una política es la regla que especifica cómo actua el agente cuando se encuentra en un estado.

Hay dos formas comunes de describirla:

- Política estocástica: asigna probabilidades a las acciones posibles. Formalmente,

$$
\pi(a\mid s)=\mathbb{P}(A_t=a \mid S_t=s).
$$

Esto significa que, estando en el estado $s$, el agente elige la acción $a$ con cierta probabilidad (no siempre hace lo mismo).

- Política determinista: en cada estado se elige una única acción. Se escribe como

$$
a=\pi(s).
$$

En este caso la regla es directa: dado $s$, la política regresa una acción especifica.

En RL, “aprender” significa encontrar una política $\pi$ que produzca buen desempeño en el entorno. En otras palabras, se busca una regla de decisión que, al repetirse a lo largo del tiempo, tienda a llevar al agente hacia mejores resultados (por ejemplo, mayor recompensa total, menos errores, o mayor probabilidad de exito).

## Retorno y objetivo

Cuando el agente interactua con el entorno, en cada paso recibe recompensas. Si a partir del tiempo $t$ el agente recibe

$$
R_{t+1}, R_{t+2}, R_{t+3}, \dots,
$$

entonces se define el retorno (también llamado ganancia acumulada) desde el tiempo $t$ como:

$$
G_t = \sum_{k=0}^{\infty}\gamma^k\,R_{t+k+1}.
$$

Aquí $\gamma \in [0,1)$ es el factor de descuento. La idea es que una recompensa que llega mas tarde cuenta menos que una recompensa inmediata. Por ejemplo, si $\gamma$ es cercano a 1, el agente le da importancia a recompensas futuras; si $\gamma$ es mas pequeño, el agente se enfoca mucho mas en lo inmediato.

En general, $G_t$ es una forma de sumar “todo lo que pase después” pero con pesos que van bajando con el tiempo. Esto ayuda a que la suma sea finita y tambien refleja que, en muchos problemas, conviene preferir resultados buenos pronto.

**Objetivo de aprendizaje**

El objetivo en RL es maximizar el retorno esperado. En términos matemáticos, dado una política $\pi$, se considera el valor esperado

$$
\mathbb{E}_\pi[G_0],
$$

donde la esperanza se toma respecto a la aleatoriedad del entorno y, si la política es estocástica, tambien respecto a las acciones que el agente elige.

Entonces se busca una política óptima $\pi^*$ que maximice ese valor esperado:

$$
\pi^* \in \arg\max_{\pi}\, \mathbb{E}_\pi[G_0].
$$

En palabras: el agente debe aprender a tomar decisiones que produzcan buenas consecuencias acumuladas a lo largo del tiempo, no solo un acierto inmediato. Esto es importante porque muchas veces una accion puede dar una recompensa pequeña hoy, pero llevar a una mejor situación despues, y eso termina siendo mejor en el total.

## Funciones de valor (cómo medimos qué tan “bueno” es un estado o una acción)

Dada una política $\pi$, se usan funciones que sirven para cuantificar qué tan conveniente es estar en un estado, o tomar cierta acción. En RL esto es clave por que casi todo se termina reduciendo a comparar “qué conviene mas”.

Se definen dos funciones principales:

**Valor de estado**

$$
V^\pi(s) = \mathbb{E}_\pi\big[G_t \mid S_t=s\big].
$$

Interpretación: si en el tiempo $t$ el agente está en el estado $s$ y a partir de ahi sigue la política $\pi$, entonces $V^\pi(s)$ dice cuanta recompensa (retorno) se espera acumular en promedio. O sea, mide lo “bueno” que es caer en $s$ cuando se actua con $\pi$.

**Valor acción–estado**

$$
Q^\pi(s,a) = \mathbb{E}_\pi\big[G_t \mid S_t=s, A_t=a\big].
$$

Interpretación: si el agente está en $s$, toma la acción $a$ ahora (en el tiempo $t$), y despues de eso sigue con la política $\pi$, entonces $Q^\pi(s,a)$ dice qué retorno se espera. En otras palabras, esto mide qué tan buena idea es hacer $a$ en $s$ si luego sigues actuando “como siempre”.

La diferencia practica es:

- $V^\pi(s)$ evalua el estado en general (sin fijar la primera acción).
- $Q^\pi(s,a)$ evalua una decisión especifica dentro del estado.

Estas funciones son como el puente entre la definición matematica del problema y los algoritmos que se implementan, por que muchos métodos aprenden o aproximan $V$ o $Q$ para decidir mejor.

## En que sentido esto se relaciona con razonamiento

Muchas tareas de razonamiento se parecen bastante a un problema de decisiones secuenciales:

- hay que elegir pasos intermedios que tengan sentido,
- se pueden cometer decisiones inutiles y eso puede costar (por ejemplo, tiempo, pasos extra o penalización),
- y muchas veces el exito real se mide hasta el final (resolver el acertijo, llegar a la meta, cumplir restricciones, etc.).

En este tipo de tareas no basta con “dar una respuesta final”, por que lo que importa es el proceso completo que te lleva ahi. Si los pasos intermedios estan mal, aunque de casualidad a veces se llegue, el desempeño promedio suele ser malo.

Por eso, en este proyecto se trabajará con un entorno tipo “acertijo”, donde el agente debe completar una secuencia de acciones correcta para recibir la recompensa principal. La idea es que el agente vaya aprendiendo qué pasos valen la pena y cuales no, y que con eso aumente la probabilidad de exito al final del episodio.

## Del razonamiento secuencial a los modelos de lenguaje

Hasta este punto, la relación entre aprendizaje por refuerzo y razonamiento puede entenderse de la siguiente manera: en ambos casos aparece una secuencia de decisiones donde los pasos intermedios importan, no solo el resultado final. Sin embargo, cuando se trabaja con modelos de lenguaje grandes, el problema adquiere una forma un poco distinta.

Un modelo de lenguaje no se mueve físicamente en un entorno ni manipula objetos materiales. Lo que produce es una secuencia de tokens, es decir, una respuesta construida paso a paso. Aun así, desde un punto de vista matemático, esa generación secuencial también puede interpretarse como un proceso de decisiones: en cada instante el modelo elige qué token producir a continuación condicionado por el contexto previo.

Dicho de otra forma, una respuesta generada por un modelo de lenguaje puede verse como una trayectoria compuesta por muchas elecciones sucesivas. Si esa respuesta busca resolver un problema, justificar una conclusión o desarrollar una cadena de razonamiento, entonces la calidad del resultado no depende solo del último token emitido, sino de toda la secuencia producida.

## Predicción del siguiente token y sus limitaciones

Los modelos de lenguaje autoregresivos se entrenan, en una primera etapa, para predecir el siguiente token de una secuencia. Ese objetivo ha demostrado ser extraordinariamente poderoso, ya que permite aprender regularidades sintácticas, semánticas e incluso ciertos patrones que parecen razonamiento. Sin embargo, este criterio de entrenamiento tiene una limitación importante.

Predecir correctamente el siguiente token no es exactamente lo mismo que optimizar una conducta compleja orientada a una meta externa. Un modelo puede aprender a producir texto gramaticalmente correcto, fluido e incluso convincente, pero eso no garantiza por sí mismo que sus respuestas sean:

- lógicamente correctas,
- útiles para resolver problemas,
- consistentes con ciertas restricciones,
- o preferibles según un criterio de calidad definido por un evaluador.

En otras palabras, el preentrenamiento enseña al modelo a continuar texto de manera plausible, pero no necesariamente a comportarse de acuerdo con objetivos especializados como razonar mejor, seguir instrucciones difíciles o maximizar una medida externa de calidad.

Debido a esa diferencia entre "predecir texto plausible" y "producir respuestas deseables según un objetivo específico", en la práctica moderna suele distinguirse entre dos momentos del desarrollo de un modelo de lenguaje:

1. **Preentrenamiento**, donde el modelo aprende patrones generales del lenguaje a gran escala.
2. **Post-entrenamiento** o ajuste posterior, donde el modelo se adapta para exhibir comportamientos más alineados con una tarea, una preferencia o un criterio de evaluación concreto.

Esta segunda etapa es especialmente importante cuando se quiere que el modelo no solo escriba texto coherente, sino que además:

- siga instrucciones,
- mantenga un formato particular,
- produzca respuestas verificables,
- o mejore su desempeño en tareas que exigen varios pasos de razonamiento.

Por tanto, cuando se habla de mejorar las capacidades de razonamiento de un modelo de lenguaje, normalmente no basta con mirar únicamente el preentrenamiento; también es necesario estudiar las técnicas de post-entrenamiento.

Desde una perspectiva matemática, el post-entrenamiento puede entenderse como un proceso donde ya no solo interesa modelar la distribución del texto, sino optimizar una señal externa de calidad.

Supóngase que, dado un prompt $x$, el modelo genera una respuesta $y$. En lugar de evaluar únicamente si $y$ coincide con una continuación observada en un corpus, ahora se introduce una función de evaluación o recompensa, que puede escribirse de manera abstracta como

$$
r(x,y).
$$

Aquí, $r(x,y)$ mide qué tan buena es la respuesta $y$ para el prompt $x$ según algún criterio: exactitud, formato, preferencia humana, verificabilidad, utilidad o corrección matemática.

Entonces el problema deja de ser solamente "ajustar el modelo para predecir el siguiente token" y pasa a ser también "ajustar el modelo para que genere respuestas que maximicen una recompensa esperada". Esta manera de formular el problema acerca de forma natural el post-entrenamiento de modelos de lenguaje al marco del aprendizaje por refuerzo.

## Cómo aparece el aprendizaje por refuerzo en modelos de lenguaje

La conexión con aprendizaje por refuerzo surge cuando se interpreta la generación de una respuesta como una secuencia de acciones. En esta interpretación:

- el estado puede entenderse como el prompt junto con los tokens ya generados;
- la acción es el siguiente token que el modelo decide emitir;
- la política es la distribución del modelo sobre posibles siguientes tokens;
- la recompensa evalúa la calidad de la respuesta generada.

Bajo esta lectura, un modelo de lenguaje define una política

$$
\pi_\theta(a \mid s),
$$

donde $s$ representa el contexto disponible y $a$ el siguiente token. El objetivo ya no es solo ajustar $\theta$ para imitar datos, sino para que la política produzca secuencias completas con mayor recompensa esperada.

Esta formulación no significa que el modelo de lenguaje sea idéntico a un agente clásico que se mueve en un laberinto o juega ajedrez. Lo que significa es que ambos problemas comparten una estructura común: una política toma decisiones secuenciales y esas decisiones pueden evaluarse mediante una señal acumulativa de calidad.

## Recompensa en tareas de razonamiento con modelos de lenguaje

En tareas de razonamiento, la recompensa no siempre se obtiene token por token. Con frecuencia, la respuesta completa se evalúa al final. Por ejemplo, una respuesta puede recibir una recompensa alta si:

- llega al resultado correcto,
- respeta un formato solicitado,
- contiene una estructura verificable,
- o satisface restricciones adicionales.

Esto introduce una dificultad importante: el modelo debe aprender a generar una secuencia larga de decisiones aunque la señal de calidad pueda aparecer hasta el final. Precisamente por eso el aprendizaje por refuerzo resulta relevante en este contexto: permite formular el ajuste del modelo en términos de maximización de una recompensa esperada, incluso cuando la recompensa es escasa, diferida o depende del resultado global.

En consecuencia, cuando se busca mejorar el razonamiento de un modelo de lenguaje, una idea natural consiste en definir funciones de recompensa adecuadas y utilizar algoritmos de aprendizaje por refuerzo para favorecer las respuestas que obtienen mejor evaluación.

## De la supervisión al ajuste por preferencias y recompensas

En el desarrollo reciente de modelos de lenguaje ha sido común distinguir entre varias etapas de ajuste. Primero, puede realizarse un ajuste supervisado con ejemplos de instrucciones y respuestas deseables. Después, puede añadirse una etapa adicional donde el modelo ya no solo imita respuestas, sino que se optimiza para producir salidas preferibles según un evaluador.

Esa evaluación puede provenir de distintas fuentes:

- juicios humanos,
- reglas automáticas,
- verificadores simbólicos,
- funciones que premian formato correcto,
- o funciones que comprueban si una solución matemática coincide con la respuesta esperada.

En todos esos casos aparece la misma idea general: existe una señal externa que permite comparar respuestas y orientar el comportamiento del modelo. Esta es justamente la clase de escenario en la que las herramientas del aprendizaje por refuerzo adquieren un papel central.

El aprendizaje por refuerzo clásico proporciona el lenguaje matemático fundamental: estados, acciones, políticas, retornos y recompensas. Sin embargo, cuando el agente es un modelo de lenguaje grande, aparecen varios elementos nuevos que no estaban presentes en los ejemplos más elementales.

Entre ellos están:

- espacios de acción extremadamente grandes, pues cada acción posible corresponde a un token del vocabulario;
- Trayectorias de longitud variable, ya que una respuesta puede tener distinta cantidad de tokens;
- recompensas definidas sobre respuestas completas;
- necesidad de mantener estabilidad durante el ajuste de modelos muy grandes;
- uso de técnicas de ajuste eficiente, como adaptaciones de bajo rango, para evitar modificar todos los parámetros del modelo.

Por esta razón, para pasar del marco clásico al caso de modelos de lenguaje, es necesario estudiar métodos de post-entrenamiento específicamente diseñados para este tipo de arquitectura y este tipo de objetivo.